In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.utils import resample
from joblib import Parallel, delayed
import plotly.graph_objects as go
import copy

from imports import *
from config import dir_config, ephys_config
from src.utils import ephys_utils

In [ ]:
compiled_dir = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)

## Utils

In [ ]:
def get_neuron_condition_trials(sessions, trial_info):
    """
    Optimized function to get neuron condition trials with reduced function calls.
    """
    neuron_condition_dict = {}
    conditions_list = [
        ("coh_0_choice_toRF_corr", (0, 1)),
        ("coh_6_choice_toRF_corr", (0.06, 1, 1)),
        ("coh_20_choice_toRF_corr", (0.2, 1, 1)),
        ("coh_50_choice_toRF_corr", (0.5, 1, 1)),
        ("coh_0_choice_awayRF_corr", (0, 0)),
        ("coh_6_choice_awayRF_corr", (0.06, 0, 1)),
        ("coh_20_choice_awayRF_corr", (0.2, 0, 1)),
        ("coh_50_choice_awayRF_corr", (0.5, 0, 1)),
    ]

    for session_id in sessions:
        neuron_ids = neuron_metadata.neuron_id[neuron_metadata.session_id == session_id].values
        conditions = {key: np.array(ephys_utils.get_trial_num(trial_info[session_id], *vals)) for key, vals in conditions_list}
        # Print conditions with no trials
        for cond_name, trials in conditions.items():
            if len(trials) == 0:
                print(f"Session {session_id} | Condition '{cond_name}' has 0 trials")
        for neuron_id in neuron_ids:
            neuron_condition_dict[neuron_id] = conditions.copy()  # Use copy to prevent reference issues

    return neuron_condition_dict


def create_pca_matrix_by_condition(neuron_condition_dict, normalize=True):
    conditions = neuron_condition_dict[list(neuron_condition_dict.keys())[0]].keys()
    PCA_data = {event: [] for event in ephys_config["alignment_settings_GP"].keys()}
    condition_len = {event: {} for event in ephys_config["alignment_settings_GP"].keys()}

    for alignment in ephys_config["alignment_settings_GP"].keys():
        for condition in conditions:
            time_duration = ephys_config["alignment_settings_GP"][alignment]["end_time_ms"] - ephys_config["alignment_settings_GP"][alignment]["start_time_ms"] + 1
            all_neuron_data = np.full((len(neuron_condition_dict.keys()), time_duration), np.nan)
            for neuron_idx, neuron_id in enumerate(neuron_condition_dict.keys()):
                trials = neuron_condition_dict[neuron_id][condition]
                condition_data = ephys_utils.get_neural_data_from_trial_num(ephys[alignment][neuron_id], trials, type="convolved_spike_trains")
                if alignment == "cue":
                    non_nan_50_prct_timepoint = np.where(np.sum(np.isnan(condition_data), axis=0) / len(trials) > 0.5)[0]
                    if non_nan_50_prct_timepoint.size == 0:
                        non_nan_50_prct_timepoint = condition_data.shape[1]
                    else:
                        non_nan_50_prct_timepoint = non_nan_50_prct_timepoint[0] - 1

                    neuron_array = condition_data[:, :non_nan_50_prct_timepoint]
                    all_neuron_data[neuron_idx, :non_nan_50_prct_timepoint] = np.nanmean(neuron_array, axis=0)

                elif alignment == "response":
                    non_nan_50_prct_timepoint = np.where(np.sum(np.isnan(condition_data), axis=0) / len(trials) > 0.5)[0]
                    if non_nan_50_prct_timepoint.size == 0:
                        non_nan_50_prct_timepoint = 0
                    else:
                        non_nan_50_prct_timepoint = non_nan_50_prct_timepoint[-1] + 1
                    neuron_array = condition_data[:, non_nan_50_prct_timepoint:]
                    all_neuron_data[neuron_idx, non_nan_50_prct_timepoint:] = np.nanmean(neuron_array, axis=0)
                else:
                    all_neuron_data[neuron_idx, :] = np.nanmean(condition_data, axis=0)

            all_non_nan_mask = ~np.any(np.isnan(all_neuron_data), axis=0)
            all_neuron_data = all_neuron_data[:, all_non_nan_mask]
            PCA_data[alignment].append(all_neuron_data)
            if all_neuron_data.shape[1] == 0:
                print(f"Warning: No data available for alignment {alignment} and condition {condition}.")
                pass
            condition_len[alignment][condition] = all_neuron_data.shape[1]

        PCA_data[alignment] = np.hstack(PCA_data[alignment])

        if normalize:
            if PCA_data[alignment].T.shape[0] > 0:
                scaler = StandardScaler()
                PCA_data[alignment] = scaler.fit_transform(PCA_data[alignment].T).T
            else:
                # print(PCA_data[alignment])
                raise ValueError(f"No data available for alignment {alignment} after normalization.")


    return PCA_data, condition_len

def go_scatter(fig, x, y, z=None, name=" ", color=None, markersize=2, linestyle="solid", linewidth=2, mode="lines+markers", opacity=0.8, showlegend=False, symbol="circle"):

    marker_dict=dict(size=markersize, color=color if color is not None else "black", opacity=opacity, symbol=symbol)
    if z is None:
        fig.add_trace(go.Scatter(x=x, y=y, mode=mode, marker=marker_dict, line=dict(width=linewidth, dash=linestyle), opacity=opacity, name=name, showlegend=showlegend))
    else:
        fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode=mode, marker=marker_dict, line=dict(width=linewidth, dash=linestyle), opacity=opacity, name=name, showlegend=showlegend))

# def go_scatter(fig, x, y, z=None, name=" ", color=None, markersize=2, linestyle="solid", linewidth=2, mode="lines+markers", opacity=0.8, showlegend=False):
#     """Helper function to add a 2D or 3D scatter plot to a Plotly figure."""
#     if z is None:
#         fig.add_trace(go.Scatter(x=x, y=y, mode=mode, marker=dict(size=markersize, color=color if color is not None else "black", opacity=opacity), line=dict(width=linewidth, dash=linestyle), opacity=opacity, name=name, showlegend=showlegend))
#     else:
#         fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode=mode, marker=dict(size=markersize, color=color if color is not None else "black", opacity=opacity), line=dict(width=linewidth, dash=linestyle), opacity=opacity, name=name, showlegend=showlegend))


def plot_pca_projection(fig, conditions, proj_mean, timepoints_list, onset_time, axes=(0, 1)):
    """
    Function to plot PCA projections with SEM as shaded areas.
    Now supports 2D and 3D plotting by choosing the number of axes.
    """

    toRF_colors = ["#96D6EC", "#6FC3EB", "#5289C6", "#4469B1"]
    awayRF_colors = ["#F2A448", "#EF8D41", "#EC6A50", "#AC3626"]

    condition_dict = {
        "coh_0_choice_toRF_corr": {"index": 0, "color": toRF_colors[0], "lw": 3, "opacity": 1},
        "coh_6_choice_toRF_corr": {"index": 1, "color": toRF_colors[1], "lw": 3, "opacity": 1},
        "coh_20_choice_toRF_corr": {"index": 2, "color": toRF_colors[2], "lw": 3, "opacity": 1},
        "coh_50_choice_toRF_corr": {"index": 3, "color": toRF_colors[3], "lw": 3, "opacity": 1},
        "coh_0_choice_awayRF_corr": {"index": 4, "color": awayRF_colors[0], "lw": 3, "opacity": 1},
        "coh_6_choice_awayRF_corr": {"index": 5, "color": awayRF_colors[1], "lw": 3, "opacity": 1},
        "coh_20_choice_awayRF_corr": {"index": 6, "color": awayRF_colors[2], "lw": 3, "opacity": 1},
        "coh_50_choice_awayRF_corr": {"index": 7, "color": awayRF_colors[3], "lw": 3, "opacity": 1},
    }

    
    for condition in conditions:
        condition_params = condition_dict[condition]  # Extract plotting parameters
        # condition = condition_params["index"]
        name = f"{condition}"
        # Select mean and parameters
        mean = proj_mean[condition]
        color = condition_params["color"]
        linewidth = condition_params["lw"]

        # Plot using the selected axes
        if len(axes) == 2:
            go_scatter(fig, mean[axes[0]], mean[axes[1]], name=name, color=color, mode="lines", linewidth=linewidth, opacity=condition_params["opacity"], showlegend=True)
        elif len(axes) == 3:
            go_scatter(fig, mean[axes[0]], mean[axes[1]], mean[axes[2]], name=name, color=color, mode="lines", linewidth=linewidth, opacity=condition_params["opacity"], showlegend=True)

    # Highlight key time points
    for idx, timepoint in enumerate(timepoints_list):
        for condition in conditions:
            condition_params = condition_dict[condition]

            # marker_size = 8 if idx == 0 else 5
            marker_size = 5
            if ~np.all(np.isnan(proj_mean[condition][:, timepoint])):
                x, y = proj_mean[condition][axes[0], timepoint], proj_mean[condition][axes[1], timepoint]
                z = proj_mean[condition][axes[2], timepoint] if len(axes) == 3 else None
                go_scatter(fig, [x], [y], [z] if z is not None else None, color=condition_params["color"], markersize=marker_size, mode="markers")
            
    # Mark onset time with violet dots
    for condition in conditions:
        condition_params = condition_dict[condition]
        # condition = condition_params["index"]

        
        if ~np.all(np.isnan(proj_mean[condition][:, onset_time])):
            x, y = proj_mean[condition][axes[0], onset_time], proj_mean[condition][axes[1], onset_time]
            z = proj_mean[condition][axes[2], onset_time] if len(axes) == 3 else None
            go_scatter(fig, [x], [y], [z] if z is not None else None,  color=condition_params["color"], markersize=10, mode="markers", symbol="square")
        

    # Get range for isometric aspect ratio
    all_vals = []
    for condition in conditions:
        all_vals.extend(proj_mean[condition][list(axes), :].flatten())
    min_val, max_val = np.nanmin(all_vals), np.nanmax(all_vals)

    print(f"PCA plot ranges: min={min_val}, max={max_val}")

    fig.update_layout(
        template="plotly_white",
        scene=dict(
            xaxis=dict(title=f"PC{axes[0]+1}", range=[min_val, max_val], backgroundcolor="white", showgrid=False, zeroline=True, showticklabels=False, ticks=""),
            yaxis=dict(title=f"PC{axes[1]+1}", range=[min_val, max_val], backgroundcolor="white", showgrid=False, zeroline=True, showticklabels=False, ticks=""),
            zaxis=dict(title=f"PC{axes[2]+1}", range=[min_val, max_val], backgroundcolor="white", showgrid=False, zeroline=True, showticklabels=False, ticks=""),
            # xaxis=dict(title=f"PC{axes[0]+1}", range=[min_val, max_val], backgroundcolor="white", showgrid=True, zeroline=False, showticklabels=False, ticks=""),
            # yaxis=dict(title=f"PC{axes[1]+1}", range=[min_val, max_val], backgroundcolor="white", showgrid=True, zeroline=False, showticklabels=False, ticks=""),
            # zaxis=dict(title=f"PC{axes[2]+1}", range=[min_val, max_val], backgroundcolor="white", showgrid=True, zeroline=False, showticklabels=False, ticks=""),
            # xaxis=dict(title=f"PC{axes[0]+1}", range=[min_val, max_val], backgroundcolor="white", linewidth=6, linecolor="black", ticks="outside"),
            # yaxis=dict(title=f"PC{axes[1]+1}", range=[min_val, max_val], backgroundcolor="white", linewidth=6, linecolor="black", ticks="outside"),
            # zaxis=dict(title=f"PC{axes[2]+1}", range=[min_val, max_val], backgroundcolor="white", linewidth=6, linecolor="black", ticks="outside"),
            aspectmode="cube" if len(axes) == 3 else None,
        ),
        width=800, height=600
    )


def plot_pca_projection_and_sem(fig, conditions, state_values, proj_biased_mean, proj_unbiased_mean, proj_biased_std, proj_unbiased_std, timepoints_list, onset_time, axes=(0, 1)):
    """
    Function to plot PCA projections with Mean and Std as shaded areas.
    Supports both 2D and 3D plotting.
    """
    condition_dict = {
        "coh_0_choice_toRF_corr": {"color": "blue", "opacity": 1},
        "coh_6_choice_toRF_corr": {"color": "green", "opacity": 1},
        "coh_20_choice_toRF_corr": {"color": "orange", "opacity": 1},
        "coh_50_choice_toRF_corr": {"color": "red", "opacity": 1},
        "coh_0_choice_awayRF_corr": {"color": "blue", "opacity": 0.3},
        "coh_6_choice_awayRF_corr": {"color": "green", "opacity": 0.3},
        "coh_20_choice_awayRF_corr": {"color": "orange", "opacity": 0.3},
        "coh_50_choice_awayRF_corr": {"color": "red", "opacity": 0.3},
    }

    for condition in conditions:
        condition_params = condition_dict[condition]
        color = condition_params["color"]
        opacity = condition_params["opacity"]

        for state in state_values:
            name = f"{condition}_{state}"

            if state == "biased_state":
                mean = proj_biased_mean[condition]
                std = proj_biased_std[condition]
            else:
                mean = proj_unbiased_mean[condition]
                std = proj_unbiased_std[condition]
            if len(axes) == 2:
                valid_mask = ~np.isnan(mean[axes[0]]) & ~np.isnan(std[axes[0]]) & ~np.isnan(mean[axes[1]]) & ~np.isnan(std[axes[1]])
                # Plot mean
                go_scatter(fig, mean[axes[0]][valid_mask], mean[axes[1]][valid_mask], name=name, color=color, mode="lines", linewidth=2, opacity=opacity, showlegend=True)
                # Plot standard deviation as shaded area
                x_valid = np.concatenate([(mean[axes[0]][valid_mask] + std[axes[0]][valid_mask] * 5), (mean[axes[0]][valid_mask] - std[axes[0]][valid_mask] * 5)[::-1]])
                y_valid = np.concatenate([(mean[axes[1]][valid_mask] + std[axes[1]][valid_mask] * 5), (mean[axes[1]][valid_mask] - std[axes[1]][valid_mask] * 5)[::-1]])
                fig.add_trace(go.Scatter(x=x_valid, y=y_valid, fill="toself", fillcolor=color, opacity=0.2, line=dict(width=0), showlegend=False, name=f"{name}_std"))

            elif len(axes) == 3:
                # Plot mean
                go_scatter(fig, mean[axes[0]], mean[axes[1]], mean[axes[2]], name=name, color=color, mode="lines", linewidth=2, opacity=opacity, showlegend=True)
                # Plot standard deviation as lines (shading is not directly possible in 3D)
                for std_offset in [-1, 1]:
                    # Offset for all three axes
                    go_scatter(
                        fig,
                        mean[axes[0]] + std_offset * std[axes[0]],  # X with std
                        mean[axes[1]] + std_offset * std[axes[1]],  # Y with std
                        mean[axes[2]] + std_offset * std[axes[2]],  # Z with std
                        name=f"{name}_std",
                        color=color,
                        mode="lines",
                        opacity=0.2,
                        showlegend=False,
                    )

    # Highlight key time points
    for idx, timepoint in enumerate(timepoints_list):
        for condition in conditions:
            condition_params = condition_dict[condition]

            for state in state_values:
                marker_size = 8 if idx == 0 else 5
                if state == "biased_state":
                    if ~np.all(np.isnan(proj_biased_mean[condition][:, timepoint])):
                        x, y = proj_biased_mean[condition][axes[0], timepoint], proj_biased_mean[condition][axes[1], timepoint]
                        z = proj_biased_mean[condition][axes[2], timepoint] if len(axes) == 3 else None
                        go_scatter(fig, [x], [y], [z] if z is not None else None, color="black", markersize=marker_size, mode="markers")
                else:
                    if ~np.all(np.isnan(proj_unbiased_mean[condition][:, timepoint])):
                        x, y = proj_unbiased_mean[condition][axes[0], timepoint], proj_unbiased_mean[condition][axes[1], timepoint]
                        z = proj_unbiased_mean[condition][axes[2], timepoint] if len(axes) == 3 else None
                        go_scatter(fig, [x], [y], [z] if z is not None else None, color="black", markersize=marker_size, mode="markers")

    # Mark onset time with violet dots
    for condition in conditions:
        for state in state_values:
            if state == "biased_state":
                mean = proj_biased_mean[condition]
            else:
                mean = proj_unbiased_mean[condition]

            if ~np.all(np.isnan(mean[:, onset_time])):
                x, y = mean[axes[0], onset_time], mean[axes[1], onset_time]
                z = mean[axes[2], onset_time] if len(axes) == 3 else None
                go_scatter(fig, [x], [y], [z] if z is not None else None, color="violet", markersize=10, mode="markers")

    go_scatter(fig, [None], [None], [None] if len(axes) == 3 else None, name="Onset Time", color="violet", markersize=10, mode="markers", showlegend=True)
    go_scatter(fig, [None], [None], [None] if len(axes) == 3 else None, name=f"{timepoints_list[1] - timepoints_list[0]}ms spacing", color="black", markersize=8, mode="markers", showlegend=True)


def bootstrap_iteration(idx_bootstrap, neuron_condition_dict, pc_weights, normalize):
    """
    Optimized bootstrap iteration using NumPy and vectorized operations.
    """
    bootstrapped_dict = {}
    for neuron_id, conditions in neuron_condition_dict.items():
        bootstrapped_dict[neuron_id] = {}
        for cond, trials in conditions.items():
            if len(trials) == 0:
                print(f"Neuron {neuron_id} | Condition '{cond}' has 0 trials (bootstrap {idx_bootstrap})")
            if idx_bootstrap > 0:
                bootstrapped_dict[neuron_id][cond] = resample(trials, random_state=idx_bootstrap)
            else:
                bootstrapped_dict[neuron_id][cond] = trials

    # Generate PCA data
    PCA_data, cond_len = create_pca_matrix_by_condition(bootstrapped_dict, normalize=normalize)
    pc_proj_cond = {alignment: split_array_by_cond(pc_weights[alignment][:, :3].T @ PCA_data[alignment], cond_len[alignment], alignment) for alignment in ephys_config["alignment_settings_GP"]}

    # return idx_bootstrap, pc_proj_cond
    return pc_proj_cond


def bootstrap_PCA(sessions, trial_info, pc_weights, normalize=True, n_bootstraps=1000):
    """
    Optimized parallelized PCA bootstrapping.
    """
    pc_projection = {
        "mean": {event: None for event in ephys_config["alignment_settings_GP"]},
        "bootstrap_sem": {event: {} for event in ephys_config["alignment_settings_GP"]},
        "bootstrap": {event: [] for event in ephys_config["alignment_settings_GP"]},
    }

    # Precompute neuron condition trials once to avoid redundancy
    neuron_condition_dict = get_neuron_condition_trials(sessions, trial_info)
    # Run bootstrap iterations in parallel
    results = Parallel(n_jobs=1, backend="loky", batch_size=10, verbose=True)(delayed(bootstrap_iteration)(idx, neuron_condition_dict, pc_weights, normalize) for idx in range(n_bootstraps + 1))

    # Process results efficiently
    for alignment in ephys_config["alignment_settings_GP"]:
        first_result = results[0][alignment]
        bootstrap_results = np.array([proj[alignment] for proj in results[1:]])

        pc_projection["mean"][alignment] = first_result
        pc_projection["bootstrap"][alignment] = bootstrap_results
        for condition in first_result.keys():
            pc_projection["bootstrap_sem"][alignment][condition] = stats.sem(np.array([bootstrap_results[iteration][condition] for iteration in range(len(bootstrap_results))]), axis=0, nan_policy="omit")

    return pc_projection


def split_array_by_cond(data, array_lengths, alignment):
    duration = ephys_config["alignment_settings_GP"][alignment]["end_time_ms"] - ephys_config["alignment_settings_GP"][alignment]["start_time_ms"] + 1
    split_arrays = {key: np.full((3, duration), np.nan) for key in array_lengths.keys()}
    start_idx = 0
    # print(array_lengths)
    for key, length in array_lengths.items():
        end_idx = start_idx + length
        # print(start_idx, end_idx)

        if alignment == "response":
            split_arrays[key][:, -(end_idx - start_idx) :] = data[:, start_idx:end_idx]
        else:
            split_arrays[key][:, : (end_idx - start_idx)] = data[:, start_idx:end_idx]
        start_idx = end_idx
    return split_arrays

## Load Data

In [ ]:
with open(Path(processed_dir, "glm_hmm_masked_final.pkl"), "rb") as f:
	glm_hmm = pickle.load(f)

session_metadata = pd.read_csv(Path(processed_dir, "sessions_metadata.csv"))
neuron_metadata = pd.read_csv(Path(processed_dir, "neuron_metadata.csv"))

session_to_exclude = ["210210_GP_JP","241209_GP_TZ"]
session_metadata = session_metadata[~session_metadata["session_id"].isin(session_to_exclude)]
neuron_metadata = neuron_metadata[~neuron_metadata["session_id"].isin(session_to_exclude)]

In [ ]:
with open(Path(processed_dir, "ephys_neuron_wise.pkl"), "rb") as f:
	ephys = pickle.load(f)

### Extract data from equal block only

In [ ]:
data = glm_hmm["data"]
n_trial_back = 1
equal_data = {}
for session_id in data:
    trial_data = pd.read_csv(Path(compiled_dir, session_id, f"{session_id}_trial.csv"), index_col=None)
    GP_trial_data = trial_data[trial_data.task_type == 1].reset_index(drop=True)
    # Get valid indices based on outcomes
    valid_idx = np.where(GP_trial_data.outcome >= 0)[0]
	# First valid trial considering n_trial_back
    first_trial = valid_idx[n_trial_back - 1] + 1
    reaction_time = np.array(GP_trial_data.reaction_time)[first_trial:]
    data[session_id]["reaction_time"] = reaction_time
    
for session_id in session_metadata["session_id"]:
     #flip back to toRF/awayRF for inputs in dPCA
    if session_metadata["prior_direction"][session_metadata["session_id"] == session_id].values[0] == "awayRF":
        data[session_id]["choices"] = 1 - data[session_id]["choices"]
        data[session_id]["stimulus"] = -data[session_id]["stimulus"]

    equal_data[session_id] = data[session_id][data[session_id].prob_toRF == 50]

### Preprocessing for PCA

In [ ]:
toRF_sessions = session_metadata["session_id"][session_metadata.prior_direction == "toRF"]
awayRF_sessions = session_metadata["session_id"][session_metadata.prior_direction == "awayRF"]

In [ ]:
PCA_data_all, _ = create_pca_matrix_by_condition(get_neuron_condition_trials(session_metadata["session_id"], equal_data), normalize=True)
PCA_data_toRF, _ = create_pca_matrix_by_condition(get_neuron_condition_trials(toRF_sessions,equal_data), normalize=True)
PCA_data_awayRF, _ = create_pca_matrix_by_condition(get_neuron_condition_trials(awayRF_sessions,equal_data), normalize=True)

### Compute PCA and explained variance

In [ ]:
def plot_cum_explained_variance(pc, alignment, ax):
    """
    Function to plot cumulative explained variance for PCA components.
    """
    explained_variance = pc.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance)

    ax.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='-')
    ax.set_title(f'{alignment.capitalize()} Epoch')
    # ax.set_xlabel('Number of Principal Components')
    # ax.set_ylabel('Cumulative Explained Variance Ratio')
    ax.set_xticks(range(1, len(cumulative_variance) + 1))
    ax.set_ylim(0, 1.05)
    ax.hlines(y=0.85, xmin=1, xmax=len(cumulative_variance), colors='r', linestyles='dashed')


In [ ]:
n_components = 20
pca = PCA(n_components=n_components)
toRF_pc, toRF_pc_weights = {}, {}
awayRF_pc, awayRF_pc_weights = {}, {}
all_pc, all_pc_weights = {}, {}
fig,ax = plt.subplots(3,4, figsize=(40,10),sharex=True,sharey=True)
fig.suptitle("Cumulative Explained Variance by PCA Components", fontsize=16)
for epoch_idx, alignment in enumerate(ephys_config["alignment_settings_GP"].keys()):
    toRF_pc[alignment] = pca.fit(PCA_data_toRF[alignment])
    toRF_pc_weights[alignment] = pca.fit_transform(PCA_data_toRF[alignment])
    plot_cum_explained_variance(toRF_pc[alignment], alignment, ax=ax[0,epoch_idx])
    awayRF_pc[alignment] = pca.fit(PCA_data_awayRF[alignment])
    awayRF_pc_weights[alignment] = pca.fit_transform(PCA_data_awayRF[alignment])
    plot_cum_explained_variance(awayRF_pc[alignment], alignment, ax=ax[1,epoch_idx])
    all_pc[alignment] = pca.fit(PCA_data_all[alignment])
    all_pc_weights[alignment] = pca.fit_transform(PCA_data_all[alignment])
    plot_cum_explained_variance(all_pc[alignment], alignment, ax=ax[2,epoch_idx])
    ax[1,epoch_idx].set_title('')
    ax[2,epoch_idx].set_title('')

### Project dpca axes onto pc subspace

In [ ]:
# with open(Path(processed_dir, f'dpca_results.pkl'), 'rb') as f:
#     dpca_dict = pickle.load(f)

# dpca_results = dpca_dict["results"]

In [ ]:
# choice_axis = {"toRF_prior": {}, "awayRF_prior": {}}
# time_axis = {"toRF_prior": {}, "awayRF_prior": {}}
# stimulus_axis = {"toRF_prior": {}, "awayRF_prior": {}}
# bias_axis = {"toRF_prior": {}, "awayRF_prior": {}}

# for prior_idx, prior in enumerate(["toRF_prior", "awayRF_prior"]):
#     for alignment in ephys_config["alignment_settings_GP"]:
#         choice_axis[prior][alignment] = dpca_results[prior][alignment]["model"].D['c'][:,0]
#         time_axis[prior][alignment] = dpca_results[prior][alignment]["model"].D['t'][:,0]
#         stimulus_axis[prior][alignment] = dpca_results[prior][alignment]["model"].D['s'][:,0]
#         bias_axis[prior][alignment] = dpca_results[prior][alignment]["model"].D['b'][:,0]

### bootstrap

In [ ]:
pc_projection_all = bootstrap_PCA(session_metadata["session_id"], equal_data, all_pc_weights, normalize=True, n_bootstraps=1)
pc_projection_toRF_prior = bootstrap_PCA(toRF_sessions, equal_data, toRF_pc_weights, normalize=True, n_bootstraps=1)
pc_projection_awayRF_prior = bootstrap_PCA(awayRF_sessions, equal_data, awayRF_pc_weights, normalize=True, n_bootstraps=1)

In [ ]:
# pc_projection = {
# 	"toRF_prior_sessions": pc_projection_toRF_prior, 
# 	"awayRF_prior_sessions": pc_projection_awayRF_prior,
# 	'all_sessions': pc_projection_all
# }


# with open(Path(processed_dir, "pc_projection_equal_only.pkl"), "wb") as f:
# 	pickle.dump(pc_projection, f)

## Plot neural trajectories

In [ ]:
# import pickle
# with open(Path(processed_dir, f'pc_projection.pkl'), 'rb') as f:
#     pc_projection = pickle.load(f)
# pc_projection_toRF_prior = pc_projection["toRF_prior"]
# pc_projection_awayRF_prior = pc_projection["awayRF_prior"]

In [ ]:
def add_axis_arrow(fig,x,y,z,color='red',legend=None):

    # Create the vector line
    line = go.Scatter3d(
        x=[0, x],  # Start and end x-coordinates
        y=[0, y],  # Start and end y-coordinates
        z=[0, z],  # Start and end z-coordinates
        mode='lines',
        line=dict(color=color, width=6),
        name=legend
    )

    # Create the arrowhead using a Cone
    arrow = go.Cone(
        x=[x],  # Arrowhead at the endpoint
        y=[y],
        z=[z],
        u=[x],  # Vector direction
        v=[y],
        w=[z],
        sizemode="absolute",
        sizeref=100,  # Adjust the size of the arrowhead
        anchor="tail",
        colorscale=[[0, 'red'], [1, 'red']],
        showscale=False
    )

    # Create the figure
    fig.add_trace(line)
    fig.add_trace(arrow)


In [ ]:

alignment = "cue"
session_included = 'toRF' # 'all' or 'toRF' or 'awayRF'
if session_included == 'toRF':
    pc_projection = pc_projection_toRF_prior
elif session_included == 'awayRF':
    pc_projection = pc_projection_awayRF_prior
else:
    pc_projection = pc_projection_all
mean_projection = pc_projection["mean"][alignment]
# [x_choice,y_choice,z_choice] = toRF_pc_weights[alignment][:,:3].T @ choice_axis['toRF_prior'][alignment]
# [x_stimulus,y_stimulus,z_stimulus] = toRF_pc_weights[alignment][:,:3].T @ stimulus_axis['toRF_prior'][alignment]
# [x_bias,y_bias,z_bias] = toRF_pc_weights[alignment][:,:3].T @ bias_axis['toRF_prior'][alignment]

# Define colors for each coherence level
alignment_dict = ephys_config["alignment_settings_GP"][alignment]
conditions = [
	# "coh_0_choice_toRF_corr",
	"coh_6_choice_toRF_corr",
	# "coh_20_choice_toRF_corr",
	"coh_50_choice_toRF_corr",
	# "coh_0_choice_awayRF_corr",
	"coh_6_choice_awayRF_corr",
	# "coh_20_choice_awayRF_corr",
	"coh_50_choice_awayRF_corr"
]

if alignment == "response":
    timepoints_list = np.arange(0, alignment_dict["end_time_ms"] - alignment_dict["start_time_ms"] + 1, 50)
else:
    timepoints_list=np.arange(-alignment_dict["start_time_ms"]+100, alignment_dict["end_time_ms"] - alignment_dict["start_time_ms"] + 1, 100)
onset_time = - alignment_dict["start_time_ms"]

#trim projection to onset time
# for key in mean_projection:
#     mean_projection[key] = mean_projection[key][:,- alignment_dict["start_time_ms"]:]
# onset_time = 0
# timepoints_list=np.arange(100, alignment_dict["end_time_ms"]  + 1, 100)

# Create a figure
fig = go.Figure()
plot_pca_projection(fig, conditions,mean_projection,
                    timepoints_list=timepoints_list, onset_time = onset_time,axes=(0,1,2))
# magnif = 50
# add_axis_arrow(fig,x_stimulus*magnif,y_stimulus*magnif,z_stimulus*magnif,color = 'black',legend='Stimulus Axis')
# add_axis_arrow(fig,x_choice*magnif,y_choice*magnif,z_choice*magnif,color = 'red',legend='Choice Axis')
# add_axis_arrow(fig,x_bias*magnif,y_bias*magnif,z_bias*magnif,color = 'blue',legend='Bias Axis')

# Set plot labels
fig.update_layout(width=1200, height=800, title=f"{session_included} sessions aligned to {alignment}")


fig.update_layout(
    scene=dict(
        xaxis=dict(
            title="PC1",
            tickmode="linear",  # or 'array'
            dtick=1000,          # distance between ticks
            tickfont=dict(size=12)
        ),
        yaxis=dict(
            title="PC2",
            tickmode="linear",
            dtick=1000,
            tickfont=dict(size=12)
        ),
        zaxis=dict(
            title="PC3",
            tickmode="linear",
            dtick=1000,
            tickfont=dict(size=12)
        )
    )
)
fig.show()
fig.write_html(f"PCA - {session_included} sessions aligned to {alignment}.html")


### SEM plotting

In [ ]:
# alignment = "response"
# state_values = ["biased_state", "unbiased_state"]  # States

# # Define colors for each coherence level
# alignment_dict = ephys_config["alignment_settings_GP"][alignment]
# conditions = [
# 	"coh_0_choice_toRF_corr",
# 	# "coh_6_choice_toRF_corr",
# 	# "coh_20_choice_toRF_corr",
# 	"coh_50_choice_toRF_corr",
# 	# "coh_0_choice_awayRF_corr",
# 	# "coh_6_choice_awayRF_corr",
# 	# "coh_20_choice_awayRF_corr",
# 	# "coh_50_choice_awayRF_corr"
# ]

# if alignment == "response_onset":
# 	timepoints_list = np.arange(0, alignment_dict["end_time_ms"] - alignment_dict["start_time_ms"] + 1, 50)
# else:
# 	timepoints_list = np.arange(0, alignment_dict["end_time_ms"] - alignment_dict["start_time_ms"] + 1, 100)

# # Create a figure
# fig = go.Figure()
# plot_pca_projection_and_sem(
# 	fig,
# 	conditions,
# 	state_values,
# 	pc_projection_biased_state_toRF_prior["mean"][alignment],
# 	pc_projection_unbiased_state_toRF_prior["mean"][alignment],
# 	pc_projection_biased_state_toRF_prior["bootstrap_sem"][alignment],
# 	pc_projection_unbiased_state_toRF_prior["bootstrap_sem"][alignment],
# 	timepoints_list=timepoints_list,
# 	onset_time=-alignment_dict["start_time_ms"],
# 	axes=(1, 2),
# )
# # Set plot labels
# fig.update_layout(width=1600, height=1200, scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"), title="Interactive 3D PCA Projection")

# fig.show()

## GPT Suggested Analysis

### 1. Grassman Angle

#### biased vs unbiased

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from joblib import Parallel, delayed
from scipy.linalg import subspace_angles

def get_neuron_choice_condition_trials(sessions, trial_info, choice=1):
    """
    Optimized function to get neuron condition trials with reduced function calls.
    """
    neuron_condition_dict = {}
    conditions_list = [
        ("coh_0_corr", (0, choice)),
        ("coh_6_corr", (0.06, choice, 1)),
        ("coh_20_corr", (0.2, choice, 1)),
        ("coh_50_corr", (0.5, choice, 1)),

    ]

    for session_id in sessions:
        neuron_ids = neuron_metadata.neuron_id[neuron_metadata.session_id == session_id].values
        conditions = {key: np.array(ephys_utils.get_trial_num(trial_info[session_id], *vals)) for key, vals in conditions_list}
        # Print conditions with no trials
        for cond_name, trials in conditions.items():
            if len(trials) == 0:
                print(f"Session {session_id} | Condition '{cond_name}' has 0 trials")
        for neuron_id in neuron_ids:
            neuron_condition_dict[neuron_id] = conditions.copy()  # Use copy to prevent reference issues

    return neuron_condition_dict

def grassmann_dist_from_subspace(X, Y, k=10):
    """Compute subspace angles and Grassmann distance."""
    pca_x = PCA(n_components=k)
    Bx = pca_x.fit(X.T).components_.T
    pca_y = PCA(n_components=k)
    By = pca_y.fit(Y.T).components_.T
    angles = subspace_angles(Bx, By)
    return angles, np.sqrt(np.sum(angles ** 2))

def grassmann_bootstrap_iteration(idx, biased_dict, unbiased_dict, normalize, alignments, resample_neurons=True):
    """Compute PCA and Grassmann distance in a single bootstrap iteration."""
    rng = np.random.default_rng(idx)

    # Resample neurons as blocks if enabled
    neuron_ids = list(biased_dict.keys())
    if resample_neurons and idx > 0:  # not for the original
        neuron_ids = rng.choice(neuron_ids, size=len(neuron_ids), replace=True)

    def bootstrap_conditions(neuron_condition_dict):
        boot_dict = {}
        for neuron_id, conditions in neuron_condition_dict.items():
            boot_dict[neuron_id] = {}
            for cond, trials in conditions.items():
                if len(trials) == 0:
                    continue
                if idx > 0:
                    resampled = trials[rng.integers(0, len(trials), len(trials))]
                else:
                    resampled = trials
                boot_dict[neuron_id][cond] = resampled
        return boot_dict

    # Bootstrap data
    biased_boot = bootstrap_conditions(biased_dict)
    unbiased_boot = bootstrap_conditions(unbiased_dict)

    # Compute PCA matrices by alignment
    PCA_biased, _ = create_pca_matrix_by_condition(biased_boot, normalize=normalize)
    PCA_unbiased, _ = create_pca_matrix_by_condition(unbiased_boot, normalize=normalize)

    # Compute Grassmann distances for all alignments
    result = {}
    for alignment in alignments:
        angles, gdist = grassmann_dist_from_subspace(PCA_biased[alignment], PCA_unbiased[alignment], k=10)
        result[alignment] = {'angles': angles, 'grassmann': gdist}

    return result

def grassmann_bootstrap_PCA(sessions, biased_info, unbiased_info, normalize=True, n_bootstraps=1000):
    """Optimized parallel Grassmann bootstrapping with calculation inside bootstrap."""
    alignments = ephys_config["alignment_settings_GP"]

    # Precompute neuron-condition dictionaries once
    biased_dict = get_neuron_condition_trials(sessions, biased_info)
    unbiased_dict = get_neuron_condition_trials(sessions, unbiased_info)

    # Run all iterations in parallel
    all_results = Parallel(n_jobs=-1, backend="loky", batch_size=10, verbose=5)(
        delayed(grassmann_bootstrap_iteration)(i, biased_dict, unbiased_dict, normalize, alignments)
        for i in range(n_bootstraps + 1)
    )

    # Organize results
    results = {a: {'original': None, 'bootstrapped': []} for a in alignments}
    for alignment in alignments:
        results[alignment]['original'] = all_results[0][alignment]

    for i in range(1, n_bootstraps + 1):
        for alignment in alignments:
            results[alignment]['bootstrapped'].append(all_results[i][alignment])

    return results


In [ ]:
toRF_grassmann_bootstrap_bias_state = grassmann_bootstrap_PCA(toRF_sessions, biased_state_trial_info, unbiased_state_trial_info, normalize=True, n_bootstraps=1000)
awayRF_grassmann_bootstrap_bias_state = grassmann_bootstrap_PCA(awayRF_sessions, biased_state_trial_info, unbiased_state_trial_info, normalize=True, n_bootstraps=1000)

In [ ]:
# Get number of bootstrap samples from first alignment
n_boot = len(next(iter(toRF_grassmann_bootstrap_bias_state.values()))['bootstrapped'])

x_labels = list(ephys_config["alignment_settings_GP"].keys())

# ===== toRF =====
toRF_bias_state_gdists = np.zeros(len(x_labels))
toRF_bias_state_gdists_mean = np.zeros(len(x_labels))
toRF_bias_state_gdists_lower = np.zeros(len(x_labels))
toRF_bias_state_gdists_upper = np.zeros(len(x_labels))

for idx, alignment in enumerate(x_labels):
    boot_vals = np.array([result['grassmann'] for result in toRF_grassmann_bootstrap_bias_state[alignment]['bootstrapped']])
    toRF_bias_state_gdists[idx] = toRF_grassmann_bootstrap_bias_state[alignment]['original']['grassmann']
    toRF_bias_state_gdists_mean[idx] = np.mean(boot_vals)
    toRF_bias_state_gdists_lower[idx] = np.percentile(boot_vals, 2.5)   # 2.5th percentile
    toRF_bias_state_gdists_upper[idx] = np.percentile(boot_vals, 97.5)  # 97.5th percentile

# ===== awayRF =====
awayRF_bias_state_gdists = np.zeros(len(x_labels))
awayRF_bias_state_gdists_mean = np.zeros(len(x_labels))
awayRF_bias_state_gdists_lower = np.zeros(len(x_labels))
awayRF_bias_state_gdists_upper = np.zeros(len(x_labels))

for idx, alignment in enumerate(x_labels):
    boot_vals = np.array([result['grassmann'] for result in awayRF_grassmann_bootstrap_bias_state[alignment]['bootstrapped']])
    awayRF_bias_state_gdists[idx] = awayRF_grassmann_bootstrap_bias_state[alignment]['original']['grassmann']
    awayRF_bias_state_gdists_mean[idx] = np.mean(boot_vals)
    awayRF_bias_state_gdists_lower[idx] = np.percentile(boot_vals, 2.5)
    awayRF_bias_state_gdists_upper[idx] = np.percentile(boot_vals, 97.5)

In [ ]:
def get_pca

In [ ]:

    # Bootstrap data
    biased_boot = bootstrap_conditions(biased_dict)
    unbiased_boot = bootstrap_conditions(unbiased_dict)

    # Compute PCA matrices by alignment
    PCA_biased, _ = create_pca_matrix_by_condition(biased_boot, normalize=normalize)
    PCA_unbiased, _ = create_pca_matrix_by_condition(unbiased_boot, normalize=normalize)

#### toRF choice vs awayRF choice

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from joblib import Parallel, delayed
from scipy.linalg import subspace_angles

def get_neuron_choice_condition_trials(sessions, trial_info, choice=1):
    """
    Optimized function to get neuron condition trials with reduced function calls.
    """
    neuron_condition_dict = {}
    conditions_list = [
        ("coh_0_corr", (0, choice)),
        ("coh_6_corr", (0.06, choice, 1)),
        ("coh_20_corr", (0.2, choice, 1)),
        ("coh_50_corr", (0.5, choice, 1)),
    ]

    for session_id in sessions:
        neuron_ids = neuron_metadata.neuron_id[neuron_metadata.session_id == session_id].values
        conditions = {key: np.array(ephys_utils.get_trial_num(trial_info[session_id], *vals)) for key, vals in conditions_list}
        # Print conditions with no trials
        for cond_name, trials in conditions.items():
            if len(trials) == 0:
                print(f"Session {session_id} | Condition '{cond_name}' has 0 trials")
        for neuron_id in neuron_ids:
            neuron_condition_dict[neuron_id] = conditions.copy()  # Use copy to prevent reference issues

    return neuron_condition_dict

def choice_grassmann_bootstrap_PCA(sessions, trial_info, normalize=True, n_bootstraps=1000):
    """Optimized parallel Grassmann bootstrapping with calculation inside bootstrap."""
    alignments = ephys_config["alignment_settings_GP"]

    # Precompute neuron-condition dictionaries once
    toRF_choice_dict = get_neuron_choice_condition_trials(sessions, trial_info, choice=1)
    awayRF_choice_dict = get_neuron_choice_condition_trials(sessions, trial_info, choice=0)

    # Run all iterations in parallel
    all_results = Parallel(n_jobs=-1, backend="loky", batch_size=10, verbose=5)(
        delayed(grassmann_bootstrap_iteration)(i, toRF_choice_dict, awayRF_choice_dict, normalize, alignments)
        for i in range(n_bootstraps + 1)
    )

    # Organize results
    results = {a: {'original': None, 'bootstrapped': []} for a in alignments}
    for alignment in alignments:
        results[alignment]['original'] = all_results[0][alignment]

    for i in range(1, n_bootstraps + 1):
        for alignment in alignments:
            results[alignment]['bootstrapped'].append(all_results[i][alignment])

    return results


In [ ]:
toRF_grassmann_bootstrap_choice_state = choice_grassmann_bootstrap_PCA(toRF_sessions, data_flipped, normalize=True, n_bootstraps=1000)
awayRF_grassmann_bootstrap_choice_state = choice_grassmann_bootstrap_PCA(awayRF_sessions, data_flipped, normalize=True, n_bootstraps=1000)

In [ ]:
toRF_grassmann_bootstrap_choice_state['cue']['original'].keys()

In [ ]:
# Get number of bootstrap samples from first alignment
n_boot = len(next(iter(toRF_grassmann_bootstrap_choice_state.values()))['bootstrapped'])

x_labels = list(ephys_config["alignment_settings_GP"].keys())

# ===== toRF =====
toRF_choice_state_gdists = np.zeros(len(x_labels))
toRF_choice_state_gdists_mean = np.zeros(len(x_labels))
toRF_choice_state_gdists_lower = np.zeros(len(x_labels))
toRF_choice_state_gdists_upper = np.zeros(len(x_labels))

for idx, alignment in enumerate(x_labels):
    boot_vals = np.array([result['grassmann'] for result in toRF_grassmann_bootstrap_choice_state[alignment]['bootstrapped']])
    toRF_choice_state_gdists[idx] = toRF_grassmann_bootstrap_choice_state[alignment]['original']['grassmann']
    toRF_choice_state_gdists_mean[idx] = np.mean(boot_vals)
    toRF_choice_state_gdists_lower[idx] = np.percentile(boot_vals, 2.5)   # 2.5th percentile
    toRF_choice_state_gdists_upper[idx] = np.percentile(boot_vals, 97.5)  # 97.5th percentile

# ===== awayRF =====
awayRF_choice_state_gdists = np.zeros(len(x_labels))
awayRF_choice_state_gdists_mean = np.zeros(len(x_labels))
awayRF_choice_state_gdists_lower = np.zeros(len(x_labels))
awayRF_choice_state_gdists_upper = np.zeros(len(x_labels))

for idx, alignment in enumerate(x_labels):
    boot_vals = np.array([result['grassmann'] for result in awayRF_grassmann_bootstrap_choice_state[alignment]['bootstrapped']])
    awayRF_choice_state_gdists[idx] = awayRF_grassmann_bootstrap_choice_state[alignment]['original']['grassmann']
    awayRF_choice_state_gdists_mean[idx] = np.mean(boot_vals)
    awayRF_choice_state_gdists_lower[idx] = np.percentile(boot_vals, 2.5)
    awayRF_choice_state_gdists_upper[idx] = np.percentile(boot_vals, 97.5)

In [ ]:
import matplotlib.pyplot as plt

def plot_rf(ax, x_labels, bias_gdists, bias_lower, bias_upper,
            choice_gdists, choice_lower, choice_upper,
            label_prefix, color='C0'):

    # Plot bias
    ax.plot(x_labels, bias_gdists, marker='o', markersize=8, label='biased vs unbiased subspace', color=color, linestyle='--')
    ax.fill_between(x_labels, bias_lower, bias_upper, color=color, alpha=0.3)

    # Plot choice
    ax.plot(x_labels, choice_gdists, marker='o', markersize=8, label='toRF vs awayRF choice subspace', color=color)
    ax.fill_between(x_labels, choice_lower, choice_upper, color=color, alpha=0.3)

    # X-axis ticks: match lengths
    ax.set_xticks(range(len(x_labels)))
    ax.set_xticklabels([label.capitalize() for label in x_labels])

    # Axis labels and formatting
    ax.set_ylabel("Grassmann Distance", fontsize=15, labelpad=20)
    ax.set_yticks([2.5, 3.0, 3.5, 4.0])
    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.set_xlabel("Epochs", fontsize=15)
    ax.set_title(f"{label_prefix} prior", fontsize=16)
    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Legend
    ax.legend(fontsize=12)


fig, axs = plt.subplots(1, 2, figsize=(16,6))

# Plot toRF on first subplot
plot_rf(
    ax=axs[0],
    x_labels=x_labels[1:],
    bias_gdists=toRF_bias_state_gdists[1:],
    bias_lower=toRF_bias_state_gdists_lower[1:],
    bias_upper=toRF_bias_state_gdists_upper[1:],
    choice_gdists=toRF_choice_state_gdists[1:],
    choice_lower=toRF_choice_state_gdists_lower[1:],
    choice_upper=toRF_choice_state_gdists_upper[1:],
    label_prefix='toRF',
    color='C0'
)
# ax.text(-0.08, 0.5, "(between biased and unbiased subspaces)",
#         transform=ax.transAxes, rotation=90, fontsize=12, va='center')

# Plot awayRF on second subplot
plot_rf(
    ax=axs[1],
    x_labels=x_labels[1:],
    bias_gdists=awayRF_bias_state_gdists[1:],
    bias_lower=awayRF_bias_state_gdists_lower[1:],
    bias_upper=awayRF_bias_state_gdists_upper[1:],
    choice_gdists=awayRF_choice_state_gdists[1:],
    choice_lower=awayRF_choice_state_gdists_lower[1:],
    choice_upper=awayRF_choice_state_gdists_upper[1:],
    label_prefix='awayRF',
    color='C1'
)

plt.tight_layout()
plt.show()
